# <center> <font color="#0036a3">Maestría en Inteligencia Artificial Aplicada (MNA)</font>  — Avance 6</center>

<center>

[![Materia](https://img.shields.io/badge/MATERIA-PROYECTO_INTEGRADOR-E0A800?style=for-the-badge&logoColor=white)](https://tec.mx)

</center>

<center>

[![Python](https://img.shields.io/badge/Python-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)
[![PyTorch](https://img.shields.io/badge/PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![GitHub](https://img.shields.io/badge/Repo-GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/jmtoral/proyecto_integrador_52)

</center>

## **<font color="#0036a3">Avance 6 — Análisis Cualitativo: Efecto del Image Enhancement bajo Sub/Sobre-exposición</font>**

### **<font color="#E0A800">Proyecto Integrador — TC5035.10</font>**

---

## **<center> <font color="#0036a3">Equipo 52</font> </center>**

<table style="border-collapse:collapse; width:60%; margin:auto;">
  <tr>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/elda.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>Elda Morales</strong><br><small>A00449074</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/mpgc.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>María Paula Gutiérrez</strong><br><small>A01747706</small>
    </td>
    <td align="center" style="border:none; width:33%; padding:10px;">
      <img src="https://raw.githubusercontent.com/jmtoral/proyecto_integrador_52/main/reports/jmtc_n.jpg" width="80" height="80" style="border-radius:50%; object-fit:cover; object-position:center top;"><br>
      <strong>José Manuel Toral</strong><br><small>A01122243</small>
    </td>
  </tr>
</table>

---

### Motivación

Las métricas globales (AbsRel, RMSE) promedian el error sobre los 550 fotogramas y **ocultan dónde** ayuda o estorba cada *enhancement*. En endoscopía, los dos regímenes de iluminación más problemáticos son:

- **Subexposición (*under*)** — zonas demasiado **oscuras** (cavidades profundas, regiones alejadas de la fuente de luz). La red pierde textura y tiende a colapsar la profundidad.
- **Sobreexposición (*over*)** — zonas **quemadas** por exceso de luz o reflejos especulares sobre tejido húmedo. La señal se satura y la geometría local se distorsiona.

Los métodos de *image enhancement* (Retinex, EndoLMSPEC, IAT) buscan precisamente **corregir estos extremos**. Este avance los inspecciona de forma **cualitativa** sobre el mejor modelo, en un fotograma representativo de cada régimen.

### Modelo seleccionado: **MonoIIT**

MonoIIT es el modelo con **menor error** del estudio (AbsRel 0.0554 sobre el split oficial), consistente con la referencia del Dr. Espinosa. Usa encoder **MPViT** + decoder **HR-Depth** (con módulos de atención), cargado tal cual el repositorio oficial de MonoViT.

### Qué se muestra

Para un fotograma **subexpuesto** y uno **sobreexpuesto** (seleccionados automáticamente por su brillo medio en el split), se compara, en columnas:
- El **ground truth** (profundidad real del sensor SCARED), como referencia
- La imagen **original** (`none`) y cada **enhancement** (`retinex`, `endolmspec`, `iat`), con el **mapa de profundidad** que MonoIIT predice de cada versión

Los mapas se muestran en **disparidad** (1/profundidad), la convención de los trabajos de *depth*: **lo cercano sale claro y lo lejano oscuro**. Así se observa si la corrección de iluminación **acerca la predicción al ground truth** en las zonas oscuras/quemadas, o si introduce artefactos.

In [9]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab; IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE         = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT  = BASE / "scared_raw"
    EDAM_PATH    = BASE / "Endo-Depth-and-Motion"
    LMSPEC_PATH  = BASE / "EndoLMSPEC"
    IAT_PATH     = BASE / "EndoViT"
    MONOVIT_PATH = BASE / "MonoViT"
    STTN_PATH    = BASE / "Endo-STTN"
    W            = BASE / "scared weights"
    REPO_ROOT    = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call(["git","clone","--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git", str(REPO_ROOT)])
    else:
        subprocess.check_call(["git","-C",str(REPO_ROOT),"fetch","origin"])
        subprocess.check_call(["git","-C",str(REPO_ROOT),"reset","--hard","origin/main"])
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
else:
    BASE         = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT  = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    EDAM_PATH    = Path("E:/Endo-Depth-and-Motion")
    LMSPEC_PATH  = Path("E:/EndoLMSPEC")
    IAT_PATH     = Path("E:/EndoVit")
    MONOVIT_PATH = Path("E:/MonoViT")
    STTN_PATH    = Path("E:/Endo-STTN")
    W            = BASE
    REPO_ROOT    = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    SPLIT_FILE   = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"

W_MONOIIT      = W / "monoIIT_weights" / "trained-winner-weights"
LMSPEC_WEIGHTS = LMSPEC_PATH / "checkpoint" / "main_net" / "model_256_combined_SSIM5_1.pth"
IAT_WEIGHTS    = IAT_PATH / "Endo4IE" / "best_Epoch50_laplacian_histogan_loss.pth"
STTN_WEIGHTS   = STTN_PATH / "release_model" / "pretrained_model" / "gen_00009.pth"
NPZ_CACHE      = BASE / "split_frames.npz"

def load_split(sf):
    items=[]
    with open(sf) as f:
        for line in f:
            line=line.strip()
            if not line: continue
            folder, fid, _ = line.split()
            ds, kf = folder.split("/")
            items.append(("dataset_"+ds.replace("dataset",""), "keyframe_"+kf.replace("keyframe",""), int(fid)))
    return items
SPLIT_ITEMS = load_split(SPLIT_FILE)
from collections import defaultdict
SPLIT_BY_KF = defaultdict(list)
for ds,kf,fid in SPLIT_ITEMS: SPLIT_BY_KF[(ds,kf)].append(fid)
print(f"{'Colab' if IN_COLAB else 'Local'} | split {len(SPLIT_ITEMS)} frames")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab | split 551 frames


In [10]:
import numpy as np
# Cargar imagenes + GT del npz cacheado
def _key(ds,kf,fid): return f"{ds}|{kf}|{fid}"
SPLIT_DATA = {}
_npz = np.load(NPZ_CACHE, allow_pickle=True)
for ds,kf,fid in SPLIT_ITEMS:
    k=_key(ds,kf,fid); ik,gk="img_"+k,"gt_"+k
    if ik in _npz.files:
        gt=_npz[gk] if gk in _npz.files else None
        if gt is not None and gt.size==1 and np.isnan(gt).all(): gt=None
        SPLIT_DATA[k]=(_npz[ik], gt)
def load_split_frame(ds,kf,fid):
    return SPLIT_DATA.get(_key(ds,kf,fid),(None,None))
print(f"Frames en memoria: {len(SPLIT_DATA)}")

Frames en memoria: 551


In [11]:
import torch, importlib.util as _ilu, types
import torch.nn as nn
import numpy as np
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_NETDIR = MONOVIT_PATH / "networks"
for _k in list(sys.modules):
    if _k=="networks" or _k.startswith("networks."): del sys.modules[_k]
_pkg=types.ModuleType("networks"); _pkg.__path__=[str(_NETDIR)]; sys.modules["networks"]=_pkg
def _ls(name,fn):
    sp=_ilu.spec_from_file_location(f"networks.{name}",str(_NETDIR/fn))
    m=_ilu.module_from_spec(sp); sys.modules[f"networks.{name}"]=m; sp.loader.exec_module(m); setattr(_pkg,name,m); return m
_ls("hr_layers","hr_layers.py")
_hr = _ls("hr_decoder","hr_decoder.py")
mpvit_small=_ls("mpvit","mpvit.py").mpvit_small
DepthDecoderHR = _hr.DepthDecoder

# MonoIIT usa encoder mpvit + decoder HR-Depth OFICIAL (convs.f4, X_00, attention),
# igual que MonoViT. Se carga con DepthDecoderHR() defaults (como evaluate_depth.py oficial).
enc = mpvit_small(); enc.num_ch_enc=[64,128,216,288,288]
_ed = torch.load(W_MONOIIT/"encoder.pth", map_location=DEVICE)
MH, MW = _ed.get("height",192), _ed.get("width",640)
enc.load_state_dict({k:v for k,v in _ed.items() if k in enc.state_dict()}); enc.to(DEVICE).eval()

_sd = torch.load(W_MONOIIT/"depth.pth", map_location=DEVICE)
dec = DepthDecoderHR()
_r = dec.load_state_dict(_sd, strict=False); dec.to(DEVICE).eval()
print(f"MonoIIT cargado {MH}x{MW} (HR-Depth) | missing={len(_r.missing_keys)} unexpected={len(_r.unexpected_keys)}")

import cv2, PIL.Image as pil
from torchvision import transforms
def predict_depth(img, max_depth=150.0, min_depth=0.1):
    H,W_=img.shape[:2]
    t=transforms.ToTensor()(pil.fromarray(img).resize((MW,MH),pil.LANCZOS)).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): out=dec(enc(t))
    disp=out[("disp",0)].squeeze().detach().cpu().numpy()
    sd=(1.0/max_depth)+((1.0/min_depth)-(1.0/max_depth))*disp
    sd=cv2.resize(sd,(W_,H)); return 1.0/sd

MonoIIT cargado 256x320 (HR-Depth) | missing=0 unexpected=0


In [12]:
# Cargar los enhancements (mismo codigo que el notebook principal)
import torchvision.transforms as T
subprocess.check_call([sys.executable,"-m","pip","install","-q","IQA_pytorch","path"])

def _load_endolmspec(p, device):
    _orig=sys.path.copy()
    clean=[str(p)]+[x for x in sys.path if "EndoSLAM" not in x and "endosfm" not in x.lower()
                    and "HADepth" not in x and "EndoViT" not in x and "EndoVit" not in x]
    for k in list(sys.modules):
        if k in ("utils","generator","unet") or k.startswith("utils."): del sys.modules[k]
    try:
        sys.path=clean
        sp=_ilu.spec_from_file_location("generator", p/"generator.py")
        mod=_ilu.module_from_spec(sp); sp.loader.exec_module(mod); G=mod.Generator
    finally: sys.path=_orig
    return G(n_channels=3, device=device, bilinear=False)
lmspec_net=_load_endolmspec(LMSPEC_PATH, DEVICE)
lmspec_net.load_state_dict(torch.load(LMSPEC_WEIGHTS, map_location=DEVICE)); lmspec_net.to(DEVICE).eval()

for k in list(sys.modules):
    if k=="utils" or k.startswith("utils."): del sys.modules[k]
sys.modules["imp"]=types.ModuleType("imp")
_sp=_ilu.spec_from_file_location("IAT_main_a5", IAT_PATH/"experiments"/"model"/"IAT_main.py")
_im=_ilu.module_from_spec(_sp)
if str(IAT_PATH/"experiments") not in sys.path: sys.path.insert(0,str(IAT_PATH/"experiments"))
_sp.loader.exec_module(_im)
iat_net=_im.IAT(in_dim=3, with_global=True, type="exp")
iat_net.load_state_dict(torch.load(IAT_WEIGHTS, map_location=DEVICE)); iat_net.to(DEVICE).eval()

def correct_none(img): return img
def correct_retinex(img, sigma=30):
    f=img.astype(np.float32)+1.0; r=np.zeros_like(f)
    for c in range(3):
        b=cv2.GaussianBlur(f[:,:,c],(0,0),sigma); r[:,:,c]=np.log(f[:,:,c])-np.log(b+1.0)
    r-=r.min(); return (r/(r.max()+1e-8)*255).astype(np.uint8)
def correct_endolmspec(img):
    t=T.ToTensor()(img).to(DEVICE)
    with torch.no_grad(): _,o=lmspec_net(t)
    return (o["subnet_16"][0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)
def correct_iat(img):
    t=torch.from_numpy(img.astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): _,_,e=iat_net(t)
    return (e[0].cpu().clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)

CORRECTIONS={"none":correct_none,"retinex":correct_retinex,"endolmspec":correct_endolmspec,"iat":correct_iat}
print("Enhancements:", list(CORRECTIONS.keys()))
print("(endosttn se omite en esta visualizacion por ser temporal/pesado; se puede anadir si se desea)")

Enhancements: ['none', 'retinex', 'endolmspec', 'iat']
(endosttn se omite en esta visualizacion por ser temporal/pesado; se puede anadir si se desea)


---
## Selección automática de fotogramas *under* y *over*

Para cada fotograma del split se calcula su **brillo medio** (canal L de CIELAB). El fotograma con menor brillo representa el caso **subexpuesto** y el de mayor brillo el **sobreexpuesto**. Sobre cada uno se aplican los *enhancements* y se obtiene el mapa de profundidad de MonoIIT.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# --- Seleccion automatica: frame mas oscuro (under) y mas brillante (over) por brillo medio (L de LAB) ---
def _brillo(img):
    return cv2.cvtColor(img, cv2.COLOR_RGB2LAB)[:,:,0].mean()

_cands = []
for ds,kf,fid in SPLIT_ITEMS:
    img,gt = load_split_frame(ds,kf,fid)
    if img is not None and gt is not None:   # exigir GT valido para poder mostrarlo
        _cands.append((_brillo(img), ds, kf, fid))
_cands.sort()
_under = _cands[0]; _over = _cands[-1]
print(f"UNDER (subexpuesto): {_under[1]}/{_under[2]} frame {_under[3]}  | L={_under[0]:.1f}")
print(f"OVER  (sobreexpuesto): {_over[1]}/{_over[2]} frame {_over[3]}  | L={_over[0]:.1f}")

CASOS = [("Subexpuesto (under)", _under), ("Sobreexpuesto (over)", _over)]
methods = list(CORRECTIONS.keys())

import os
os.makedirs(REPO_ROOT/"outcomes"/"avance6", exist_ok=True)

# Visualizar DISPARIDAD (1/depth): convencion de los papers de depth.
# Asi lo CERCANO sale brillante y lo LEJANO oscuro (la version previa estaba invertida).
def _pred_disp_vis(depth):
    disp = 1.0/np.clip(depth, 1e-6, None)
    return np.clip(disp, 0, np.percentile(disp, 95))

def _gt_disp_vis(gt):
    g = gt.astype(np.float32).copy(); g[~np.isfinite(g)] = np.nan
    disp = 1.0/np.clip(g, 1e-6, None)
    return np.clip(disp, 0, np.nanpercentile(disp, 95))

# Columnas: GroundTruth + cada metodo. Fila 0 = imagen, Fila 1 = disparidad.
for titulo, (_b, ds, kf, fid) in CASOS:
    img, gt = load_split_frame(ds,kf,fid)
    if img is None: continue
    ncol = 1 + len(methods)
    fig, axes = plt.subplots(2, ncol, figsize=(4*ncol, 7))
    fig.suptitle(f"{titulo}  -  MonoIIT  -  {ds}/{kf} frame {fid}  (L={_b:.0f})",
                 fontsize=13, fontweight="bold")

    # Columna 0 = Ground Truth
    axes[0,0].imshow(img); axes[0,0].set_title("imagen original", fontsize=11); axes[0,0].axis("off")
    axes[1,0].imshow(_gt_disp_vis(gt), cmap="magma")
    axes[1,0].set_title("GROUND TRUTH", fontsize=11, color="#0036a3", fontweight="bold"); axes[1,0].axis("off")
    axes[0,0].text(-0.12, 0.5, "entrada", rotation=90, va="center", ha="center",
                   transform=axes[0,0].transAxes, fontsize=10, fontweight="bold")
    axes[1,0].text(-0.12, 0.5, "profundidad\n(cerca = claro)", rotation=90, va="center", ha="center",
                   transform=axes[1,0].transAxes, fontsize=9, fontweight="bold")

    # Columnas 1..N = enhancements
    for j, m in enumerate(methods, start=1):
        corr = CORRECTIONS[m](img)
        depth = predict_depth(corr)
        axes[0,j].imshow(corr); axes[0,j].set_title(m, fontsize=11); axes[0,j].axis("off")
        axes[1,j].imshow(_pred_disp_vis(depth), cmap="magma"); axes[1,j].axis("off")

    plt.tight_layout()
    _slug = "under" if "under" in titulo.lower() else "over"
    _out = REPO_ROOT/"outcomes"/"avance6"/f"avance6_{_slug}.png"
    fig.savefig(_out, dpi=110, bbox_inches="tight")
    print(f"Guardado: {_out}")
    plt.show()

---
## Lectura de los resultados

Los mapas se muestran en **disparidad** (1/profundidad): **lo cercano es claro, lo lejano oscuro**. La columna **GROUND TRUTH** (azul) es la referencia real del sensor SCARED; cada predicción debe parecerse a ella.

**Caso subexpuesto (*under*).** En las zonas oscuras la imagen `none` aporta poca textura, por lo que el mapa de profundidad tiende a aplanarse o a colapsar. Retinex e IAT **levantan la luminancia** y recuperan estructura; conviene observar si la disparidad predicha se **acerca al ground truth** (recupera el gradiente de superficie) sin introducir ruido en las sombras.

**Caso sobreexpuesto (*over*).** En las regiones quemadas la información está saturada. Aquí interesa ver si el *enhancement* **atenúa el brillo** y devuelve geometría plausible (coherente con el GT) en las zonas especulares, o si —al contrario— amplifica artefactos. Retinex suele cambiar fuertemente el balance de color (efecto "lavado"), lo que puede o no beneficiar a la red según cómo fue entrenada.

**Relación con las métricas.** Estas observaciones cualitativas complementan la tabla del estudio: un *enhancement* puede mejorar el AbsRel **global** pero deteriorar un régimen específico (o viceversa). El análisis por exposición, contra el ground truth, ayuda a explicar *por qué* IAT tiende a ayudar (corrección suave que preserva la estructura) mientras que Retinex puede degradar (transformación agresiva del color/contraste).

> **Nota.** Los fotogramas se eligen automáticamente por brillo medio; al reejecutar sobre el mismo split son reproducibles. El *enhancement* **endosttn** (inpainting temporal de especularidades) se omite en esta vista por requerir la secuencia completa del *keyframe*; su efecto visual se documenta en el notebook del Avance 5.

In [ ]:
# --- Generar HTML de entrega embebiendo las PNG under/over (autocontenido) ---
import os, base64
_out_dir = REPO_ROOT/"docs"/"reports"; os.makedirs(_out_dir, exist_ok=True)
_img_dir = REPO_ROOT/"outcomes"/"avance6"

def _b64(p):
    with open(p,"rb") as f: return base64.b64encode(f.read()).decode()

_png_under = _img_dir/"avance6_under.png"
_png_over  = _img_dir/"avance6_over.png"

_secs = ""
for _png, _cap in [(_png_under, "Subexpuesto (under)"), (_png_over, "Sobreexpuesto (over)")]:
    if _png.exists():
        _secs += '<h2>' + _cap + '</h2>'
        _secs += '<img src="data:image/png;base64,' + _b64(_png) + '" style="width:100%;max-width:1200px;border:1px solid #ddd;border-radius:6px;margin:12px 0;">'

_html_doc = f"""<!doctype html><html lang="es"><head><meta charset="utf-8">
<title>Avance 6 - Equipo 52</title>
<style>
 body{{font-family:-apple-system,Segoe UI,Roboto,Arial,sans-serif;max-width:1280px;margin:30px auto;padding:0 18px;color:#1a1a1a;line-height:1.55}}
 h1{{color:#0036a3}} h2{{color:#0036a3;border-bottom:2px solid #E0A800;padding-bottom:4px;margin-top:34px}}
 .tag{{display:inline-block;background:#0036a3;color:#fff;padding:3px 10px;border-radius:4px;font-size:.8em;margin-bottom:10px}}
 .note{{background:#fff8e6;border-left:4px solid #E0A800;padding:10px 14px;border-radius:4px;margin:16px 0}}
 code{{background:#f0f2f5;padding:1px 5px;border-radius:3px}}
</style></head><body>
<span class="tag">Proyecto Integrador TC5035.10 - Equipo 52</span>
<h1>Avance 6 - Efecto del Image Enhancement bajo Sub/Sobre-exposicion</h1>
<p><b>Modelo:</b> MonoIIT (AbsRel 0.0554, el de menor error del estudio; encoder MPViT + decoder HR-Depth).
La primera columna es el <b>ground truth</b> (profundidad real del sensor SCARED). Las siguientes muestran
la imagen original y cada enhancement (<code>retinex</code>, <code>endolmspec</code>, <code>iat</code>),
con el mapa de profundidad que MonoIIT predice de cada una.</p>
<div class="note"><b>Como leer los mapas:</b> se muestran en <b>disparidad</b> (1/profundidad), la convencion
de los trabajos de depth: <b>lo cercano sale claro y lo lejano oscuro</b>. Los fotogramas se eligen
automaticamente por brillo medio (canal L de CIELAB): el mas oscuro (<i>under</i>) y el mas brillante (<i>over</i>).</div>
{_secs}
<h2>Lectura</h2>
<p><b>Under:</b> en zonas oscuras la imagen <code>none</code> da poca textura y la profundidad se aplana;
Retinex/IAT levantan luminancia y pueden recuperar estructura. Se compara contra el ground truth para
ver si la prediccion se acerca a la geometria real.
<b>Over:</b> en zonas quemadas la senal se satura; interesa ver si el enhancement atenua el brillo
y devuelve geometria plausible, o si amplifica artefactos.</p>
<p>Estas vistas cualitativas complementan la tabla del estudio: un enhancement puede mejorar el AbsRel
global pero deteriorar un regimen especifico (o viceversa).</p>
</body></html>"""

_html_path = _out_dir/"avance6_visualizacion.html"
with open(_html_path,"w",encoding="utf-8") as f: f.write(_html_doc)
print(f"HTML de entrega generado: {_html_path}")
print(f"  under: {'OK' if _png_under.exists() else 'FALTA'} | over: {'OK' if _png_over.exists() else 'FALTA'}")
print("Subelo al repo: git add docs/reports/avance6_visualizacion.html outcomes/avance6/")

In [15]:
import subprocess
from getpass import getpass
token = getpass("GitHub token: ")
subprocess.run(["git","-C",str(REPO_ROOT),"remote","set-url","origin",
    f"https://{token}@github.com/jmtoral/proyecto_integrador_52.git"])
subprocess.run(["git","-C",str(REPO_ROOT),"add",
    "docs/reports/avance6_visualizacion.html","outcomes/avance6"])
subprocess.run(["git","-C",str(REPO_ROOT),"commit","-m","Add: HTML y PNG del Avance6"])
subprocess.run(["git","-C",str(REPO_ROOT),"pull","--rebase","origin","main"])
subprocess.run(["git","-C",str(REPO_ROOT),"push","origin","main"])

CompletedProcess(args=['git', '-C', '/content/repo_52', 'push', 'origin', 'main'], returncode=0)